# Feature Circuits Phase A: Train SAEs on Qwen3.6 + Feature Alignment

**Context**: CREST Phase 1b is running. This is the parallel pivot — starts building **feature circuits** on Qwen3.6 using existing Stage B activations (L11/L17/L23).

**Pipeline (this notebook)**:
1. Inspect Stage B shard structure
2. Extract per-token activations at L11, L17, L23
3. Train 3 TopK SAEs (n=128, k=8) — first SAEs on Qwen3.6
4. Encode all rollouts to sparse features
5. **Co-activation Jaccard** between layer pairs → behavioral alignment
6. **Decoder-encoder cosine** between layers → geometric alignment
7. Combined edge ranking → top-1000 candidate edges per layer pair
8. Upload to HF `caiovicentino1/qwen36-feature-circuits`

**Budget**: ~30-45 min on RTX 6000. No model loading (SAE on pre-stored activations).

**Next** (Notebook B, not here): gate detection + gated edge transcoder training.

## Cell 1 — Install + setup

In [ ]:
import sys, subprocess, os
from pathlib import Path

def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)
try:
    import safetensors, huggingface_hub
except Exception:
    pip('install', '-q', 'safetensors', 'huggingface_hub==1.5.0', 'hf_transfer', 'datasets', 'pandas', 'scipy')

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi, create_repo, login
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t: login(token=t); print('HF auth OK')
except: pass

OUT = Path('/content/fc_phase_a'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/qwen36-feature-circuits'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'

# Hyperparams
LAYERS = ['l11', 'l17', 'l23']
D_MODEL = 2048
N_FEATURES = 128
TOPK = 8
SAE_EPOCHS = 30
SAE_BS = 4096
SAE_LR = 3e-4

print('setup done')

## Cell 2 — Inspect Stage B shard structure

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
print(f'{len(shards)} shards')

# Inspect first shard
s0 = shards[0]
with safe_open(str(s0), framework='pt') as f:
    keys = list(f.keys())
    meta = f.metadata()
print(f'\nFirst shard {s0.name}:')
print(f'  metadata keys: {list(meta.keys())[:10]}')
print(f'  tensor keys ({len(keys)} total), first 15:')
for k in keys[:15]:
    with safe_open(str(s0), framework='pt') as f:
        t = f.get_tensor(k)
    print(f'    {k}: {tuple(t.shape)} {t.dtype}')

## Cell 3 — Extract per-token activations into one big matrix per layer

Expected shard key pattern: `r{N}_l{L}` (rollout N, layer L) or similar. Adapt below if different.

In [ ]:
# Auto-detect key pattern and extract
def extract_layer_across_shards(shards, layer_tag, max_tokens=None):
    """Collect [N_tokens, d_model] across all shards for one layer."""
    chunks = []
    total = 0
    for sh in tqdm(shards, desc=f'extract {layer_tag}'):
        with safe_open(str(sh), framework='pt') as f:
            keys = [k for k in f.keys() if layer_tag in k.lower()]
            for k in keys:
                t = f.get_tensor(k)  # expect [n_tok, d] or [d] or [n_rollouts, n_tok, d]
                if t.dim() == 1:
                    t = t.unsqueeze(0)
                elif t.dim() == 3:
                    t = t.reshape(-1, t.shape[-1])
                chunks.append(t.to(torch.float32).cpu())
                total += t.shape[0]
        if max_tokens and total >= max_tokens: break
    X = torch.cat(chunks, dim=0)
    if max_tokens and X.shape[0] > max_tokens:
        idx = torch.randperm(X.shape[0])[:max_tokens]
        X = X[idx]
    return X

# Extract each layer. Cap at 300k tokens to keep training cheap/fast.
acts = {}
for L in LAYERS:
    X = extract_layer_across_shards(shards, L, max_tokens=300_000)
    acts[L] = X
    print(f'{L}: {tuple(X.shape)} {X.dtype}  mean_norm={X.norm(dim=-1).mean().item():.2f}')

## Cell 4 — Train TopK SAE on each layer

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k):
        super().__init__()
        self.W_enc = nn.Parameter(torch.empty(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.empty(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
        # Init: W_enc = W_dec.T scaled
        w = torch.randn(d_in, n) / (d_in ** 0.5)
        self.W_enc.data = w
        self.W_dec.data = w.T.contiguous()

    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        return z, top_i, top_v

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

    def forward(self, x):
        z, _, _ = self.encode(x)
        return self.decode(z), z

def train_sae(X, d_in, n, k, epochs, bs, lr, name=''):
    X = X.cuda()
    sae = TopKSAE(d_in, n, k).cuda()
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    N = X.shape[0]
    losses = []
    for e in range(epochs):
        perm = torch.randperm(N, device='cuda')
        el = 0; nb = 0
        for i in range(0, N, bs):
            xb = X[perm[i:i+bs]]
            xh, z = sae(xb)
            loss = F.mse_loss(xh, xb)
            opt.zero_grad(); loss.backward(); opt.step()
            el += loss.item(); nb += 1
        losses.append(el/nb)
        if (e+1) % 5 == 0 or e == 0:
            # Variance explained
            with torch.no_grad():
                xh, _ = sae(X[:10000])
                var_expl = 1 - ((xh - X[:10000]).var()/X[:10000].var()).item()
            print(f'  {name} ep{e+1:>2}/{epochs} mse={losses[-1]:.4f} var_expl={var_expl:.3f}')
    return sae.cpu(), losses

saes = {}
for L in LAYERS:
    print(f'\n=== Training SAE on {L} ===')
    sae, _ = train_sae(acts[L], D_MODEL, N_FEATURES, TOPK,
                       SAE_EPOCHS, SAE_BS, SAE_LR, name=L)
    saes[L] = sae
    torch.save(sae.state_dict(), OUT / f'sae_{L}_n{N_FEATURES}_k{TOPK}.pt')
    torch.cuda.empty_cache()

## Cell 5 — Encode all tokens → sparse feature matrix per layer

In [ ]:
# Encode acts[L] with saes[L] → binary feature fire matrix [N_tokens, n_features]
feature_fires = {}  # L -> [N, n] bool
feature_vals = {}   # L -> [N, n] float (magnitudes for top-k positions)

for L in LAYERS:
    sae = saes[L].cuda()
    X = acts[L].cuda()
    N = X.shape[0]
    fires = torch.zeros(N, N_FEATURES, dtype=torch.bool)
    vals = torch.zeros(N, N_FEATURES, dtype=torch.float16)
    bs = 8192
    with torch.no_grad():
        for i in tqdm(range(0, N, bs), desc=f'encode {L}'):
            xb = X[i:i+bs]
            z, top_i, top_v = sae.encode(xb)
            fires[i:i+bs] = (z > 0).cpu()
            vals[i:i+bs] = z.to(torch.float16).cpu()
    feature_fires[L] = fires
    feature_vals[L] = vals
    sae.cpu()
    torch.cuda.empty_cache()
    # Firing stats
    fire_rate = fires.float().mean(dim=0)
    print(f'{L}: n_features={N_FEATURES}, mean_fire_rate={fire_rate.mean()*100:.2f}%, '
          f'min={fire_rate.min()*100:.2f}%, max={fire_rate.max()*100:.2f}%')

## Cell 6 — Co-activation Jaccard between layer pairs

For each (L_A, L_B) pair and each (feature_i in A, feature_j in B), compute:
`J(i,j) = |fires_A[i] ∩ fires_B[j]| / |fires_A[i] ∪ fires_B[j]|`

Since L_A and L_B are captured at the *same token positions* (same forward), intersection makes sense.

In [ ]:
def jaccard_matrix(A, B):
    """A: [N, n1] bool, B: [N, n2] bool -> [n1, n2] float32."""
    Af = A.float().cuda()
    Bf = B.float().cuda()
    # intersection: Af.T @ Bf -> [n1, n2]
    inter = Af.T @ Bf
    fA = Af.sum(dim=0, keepdim=True).T  # [n1, 1]
    fB = Bf.sum(dim=0, keepdim=True)    # [1, n2]
    union = fA + fB - inter
    J = inter / union.clamp_min(1)
    return J.cpu()

pairs = [('l11','l17'), ('l17','l23'), ('l11','l23')]
jaccard = {}
for a, b in pairs:
    J = jaccard_matrix(feature_fires[a], feature_fires[b])
    jaccard[(a,b)] = J
    print(f'{a}->{b}: Jaccard [{N_FEATURES}x{N_FEATURES}], max={J.max().item():.3f}, '
          f'mean_top5={J.view(-1).sort(descending=True).values[:5].mean().item():.3f}')

## Cell 7 — Geometric alignment: decoder_A ↔ encoder_B

If feature i at layer A writes direction d_i and feature j at layer B reads direction e_j, then cos(d_i, e_j) tells us whether A's output lines up with B's input along a natural mechanical path.

In [ ]:
def decoder_encoder_cos(sae_A, sae_B):
    W_dec_A = sae_A.W_dec.data  # [n1, d]
    W_enc_B = sae_B.W_enc.data  # [d, n2]
    dec_norm = W_dec_A / W_dec_A.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    enc_norm = W_enc_B / W_enc_B.norm(dim=0, keepdim=True).clamp_min(1e-8)
    cos = dec_norm @ enc_norm  # [n1, n2]
    return cos

cosines = {}
for a, b in pairs:
    C = decoder_encoder_cos(saes[a], saes[b])
    cosines[(a,b)] = C
    print(f'{a}->{b}: cos [{N_FEATURES}x{N_FEATURES}], '
          f'abs_max={C.abs().max().item():.3f}, abs_mean_top5={C.abs().view(-1).sort(descending=True).values[:5].mean().item():.3f}')

## Cell 8 — Combined edge ranking + save top edges per pair

Score each candidate edge (i,j) as `score = J(i,j) * |cos(d_i, e_j)|` — both behavioral and geometric alignment.

In [ ]:
top_edges = {}
for a, b in pairs:
    J = jaccard[(a,b)]
    C = cosines[(a,b)].abs()
    score = J * C  # elementwise
    # top-K flat
    K = 200
    flat = score.view(-1)
    vals, idxs = flat.topk(K)
    n1, n2 = score.shape
    edges = []
    for v, fi in zip(vals.tolist(), idxs.tolist()):
        i = fi // n2
        j = fi % n2
        edges.append(dict(
            src_feat=int(i), dst_feat=int(j),
            score=float(v),
            jaccard=float(J[i,j]),
            cos=float(cosines[(a,b)][i,j].item())))
    top_edges[f'{a}->{b}'] = edges
    print(f'\n{a}->{b} top-10 edges:')
    for e in edges[:10]:
        print(f'  f{e["src_feat"]:>3}->f{e["dst_feat"]:>3}  score={e["score"]:.3f}  '
              f'J={e["jaccard"]:.3f}  cos={e["cos"]:+.3f}')

# Save full matrices + top edges
np.savez(OUT / 'edge_matrices.npz',
         **{f'jaccard_{a}_{b}': jaccard[(a,b)].numpy() for a,b in pairs},
         **{f'cos_{a}_{b}': cosines[(a,b)].numpy() for a,b in pairs})
with open(OUT / 'top_edges.json', 'w') as f:
    json.dump(top_edges, f, indent=2)
print('\nSaved matrices + top-200/pair.')

## Cell 9 — Upload to HF

In [ ]:
create_repo(HF_OUT, repo_type='model', private=False, exist_ok=True)
api = HfApi()

readme = f'''---
license: mit
base_model: Qwen/Qwen3.6-35B-A3B
tags:
  - sparse-autoencoder
  - feature-circuits
  - mechanistic-interpretability
  - hybrid-moe-gdn
---

# Qwen3.6-35B-A3B Feature Circuits — Phase A

First SAEs trained on Qwen3.6-35B-A3B hybrid MoE+GDN residual stream.

## Contents

- `sae_l11_n{N_FEATURES}_k{TOPK}.pt` — TopK SAE for layer 11 (reasoning zone entry)
- `sae_l17_n{N_FEATURES}_k{TOPK}.pt` — TopK SAE for layer 17
- `sae_l23_n{N_FEATURES}_k{TOPK}.pt` — TopK SAE for layer 23 (state convergence)
- `edge_matrices.npz` — full Jaccard + decoder-encoder cos matrices per layer pair
- `top_edges.json` — top-200 candidate edges per pair by combined score J·|cos|

## Training

- Source: Stage B corpus (`{HF_STAGE_B}`), ~300k tokens per layer
- Arch: TopK SAE, n_features={N_FEATURES}, k={TOPK}, d_model={D_MODEL}
- Epochs: {SAE_EPOCHS}, bs={SAE_BS}, lr={SAE_LR}

## Next (Phase B)

Gate detection (AND/OR/XOR) on top edges + gated edge transcoder training.
'''

with open(OUT / 'README.md', 'w') as f: f.write(readme)

files = (list(OUT.glob('sae_*.pt'))
         + [OUT/'edge_matrices.npz', OUT/'top_edges.json', OUT/'README.md'])
for fp in files:
    try:
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=fp.name,
                        repo_id=HF_OUT, repo_type='model',
                        commit_message='FC Phase A: ' + fp.name)
        print(f'OK {fp.name}')
    except Exception as e:
        print(f'FAIL {fp.name}: {e}')

print(f'\nOK https://huggingface.co/{HF_OUT}')